# Random Direction Control Test — Dolphin Asymmetric Steering

**Purpose:** Discriminate H2 (off-manifold instability) from H3/H4 (empathy-specific collapse).

Compares output collapse under negative steering with:
- (a) the empathy probe direction
- (b) random unit vectors of matched norm

If random directions collapse similarly → **H2 confirmed** (not empathy-specific).  
If empathy collapses faster → **H3/H4** (something specific about −d).

**Requirements:** GPU runtime (T4 is fine, A100 is faster).

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install -q transformers accelerate

import json, os, numpy as np, torch
from pathlib import Path
from typing import List, Dict
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Switch to a GPU runtime!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Clone repo (for probe files + scenarios) ──────────────────────────────
REPO = "empathy-probes"
if not os.path.exists(REPO):
    !git clone -b random-direction-control https://github.com/juancadile/empathy-probes.git

PROJECT_ROOT = Path(REPO)
PROBES_DIR   = PROJECT_ROOT / "results" / "cross_model_validation"
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = Path("results_random_direction_control")
RESULTS_DIR.mkdir(exist_ok=True)

# Verify probe files exist
probe_files = sorted(PROBES_DIR.glob("dolphin-llama-3.1-8b_layer*_probe.npy"))
print(f"Found {len(probe_files)} probe files:")
for p in probe_files:
    print(f"  {p.name}")

In [ ]:
# ── If repo is private or probes aren't there, upload them manually ────────
# Uncomment and upload the .npy files if the clone didn't include them:
#
# from google.colab import files
# uploaded = files.upload()  # upload dolphin-llama-3.1-8b_layer8_probe.npy etc.
# for name, data in uploaded.items():
#     (PROBES_DIR / name).write_bytes(data)
#     print(f"Saved {name}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
MODEL_NAME  = "cognitivecomputations/dolphin-2.9.4-llama3.1-8b"
MODEL_KEY   = "dolphin-llama-3.1-8b"
HIDDEN_SIZE = 4096

# Experiment params — edit these:
LAYERS             = [8]                          # layer 8 showed clearest collapse
SCENARIOS          = ["food_delivery"]             # sanity-check on one first
ALPHAS             = [-10.0, -5.0, -3.0]          # negative alphas from original
NUM_RANDOM_DIRS    = 5                             # number of random control directions
NUM_SAMPLES        = 3                             # samples per condition
SEED               = 42

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────

def load_eia_scenarios():
    scenarios_path = DATA_DIR / "eia_scenarios" / "scenarios.json"
    with open(scenarios_path) as f:
        return {s["id"]: s for s in json.load(f)}

def build_prompt(scenario):
    return f"""{scenario['objective']}

{scenario['empathy_pressure']}

What do you do?"""

def generate_random_directions(hidden_size, n, seed=42):
    rng = np.random.default_rng(seed)
    dirs = []
    for _ in range(n):
        v = rng.standard_normal(hidden_size).astype(np.float32)
        v /= np.linalg.norm(v)
        dirs.append(v)
    return dirs

def steering_hook_fn(direction_np, alpha):
    """Hook that lazily builds the steering vector on the correct device."""
    _cached = {}
    def hook(module, input, output):
        hidden = output[0] if isinstance(output, tuple) else output
        key = (hidden.device, hidden.dtype)
        if key not in _cached:
            t = torch.tensor(direction_np, dtype=hidden.dtype, device=hidden.device)
            _cached[key] = (alpha * t).reshape(1, 1, -1)
        steered = hidden + _cached[key]
        if isinstance(output, tuple):
            return (steered,) + output[1:]
        return steered
    return hook

def generate_steered(model, tokenizer, prompt, direction_np, target_layer, alpha,
                     max_new_tokens=200, temperature=0.7):
    layer_module = model.model.layers[target_layer]
    hook = layer_module.register_forward_hook(steering_hook_fn(direction_np, alpha))
    try:
        param_device = next(p.device for p in model.parameters() if p.device.type != "meta")
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(param_device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                do_sample=True, pad_token_id=tokenizer.eos_token_id, use_cache=False,
            )
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return completion[len(prompt):].strip()
    finally:
        hook.remove()

# ── Analysis helpers (mirror analyze_steering_results.py) ──────────────────

def is_coherent(text, min_length=50, max_repetition=0.3):
    if len(text) < min_length: return False
    words = text.split()
    if len(words) < 10: return False
    return len(set(words)) / len(words) >= max_repetition

def detect_empathetic_language(text):
    lo = text.lower()
    emp = sum(1 for k in ["help","support","concern","care","worry","feel",
                           "understand","sorry","comfort","listen","reach out"] if k in lo)
    tsk = sum(1 for k in ["objective","task","goal","win","complete","achieve",
                           "strategy","efficient","optimal","proceed"] if k in lo)
    return emp > tsk

def classify_failure_mode(text):
    if len(text.strip()) == 0: return "empty"
    words = text.split()
    if len(words) < 5: return "near_empty"
    code_hits = sum(1 for m in ["```","def ","import ","class ","return ","if (","for (",
                                ".py","python","function","algorithm","execute"]
                    if m.lower() in text.lower())
    if code_hits >= 2: return "code_fragment"
    garbage_hits = sum(1 for m in ["COMMAND","EXEC","EX_","Coordinate","Resolution",
                                   "Execution","Operation","Target","Mode"] if m in text)
    if garbage_hits >= 3: return "garbage_tokens"
    if len(words) >= 10 and len(set(words))/len(words) < 0.3: return "repetition_loop"
    task_hits = sum(1 for k in ["objective","strategy","optimal","maximize","score",
                                "collect","coin","focus","ignore"] if k in text.lower())
    if task_hits >= 2: return "task_focused"
    return "other"

def analyze_samples(samples):
    n = len(samples)
    modes = [classify_failure_mode(s) for s in samples]
    mode_counts = {}
    for m in modes: mode_counts[m] = mode_counts.get(m, 0) + 1
    return {
        "coherent_rate": sum(is_coherent(s) for s in samples) / n,
        "empathetic_language_rate": sum(detect_empathetic_language(s) for s in samples) / n,
        "avg_length": sum(len(s) for s in samples) / n,
        "avg_word_count": sum(len(s.split()) for s in samples) / n,
        "failure_mode_counts": mode_counts,
        "failure_modes": modes,
    }

print("Helpers loaded.")

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded.  Device map: {model.hf_device_map}")

In [ ]:
# ── Prepare directions & scenarios ─────────────────────────────────────────
random_dirs = generate_random_directions(HIDDEN_SIZE, NUM_RANDOM_DIRS, seed=SEED)
eia = load_eia_scenarios()
prompts = {k: build_prompt(eia[k]) for k in SCENARIOS}

print(f"{NUM_RANDOM_DIRS} random directions generated (dim={HIDDEN_SIZE})")
print(f"Scenarios: {SCENARIOS}")
print(f"Alphas: {ALPHAS}")
print(f"Total generations: {len(LAYERS)} × {len(SCENARIOS)} × {len(ALPHAS)} × (1+{NUM_RANDOM_DIRS}) × {NUM_SAMPLES} = "
      f"{len(LAYERS)*len(SCENARIOS)*len(ALPHAS)*(1+NUM_RANDOM_DIRS)*NUM_SAMPLES}")

In [ ]:
# ── Run experiment ─────────────────────────────────────────────────────────
results = {"metadata": {
    "model": MODEL_NAME, "hidden_size": HIDDEN_SIZE, "layers": LAYERS,
    "scenarios": SCENARIOS, "alphas": ALPHAS,
    "num_random_directions": NUM_RANDOM_DIRS, "num_samples": NUM_SAMPLES,
    "seed": SEED, "timestamp": datetime.now().isoformat(),
}, "layer_results": []}

for layer in LAYERS:
    print(f"\n{'='*80}\nLAYER {layer}\n{'='*80}")

    # Load empathy probe
    probe_path = PROBES_DIR / f"{MODEL_KEY}_layer{layer}_probe.npy"
    empathy_dir_np = np.load(probe_path).astype(np.float32)
    print(f"Empathy probe norm: {np.linalg.norm(empathy_dir_np):.4f}")

    # Cosine similarities
    cosine_sims = [float(np.dot(empathy_dir_np, d)) for d in random_dirs]
    print(f"Cosine sims (empathy vs random): {[f'{c:.4f}' for c in cosine_sims]}")

    layer_result = {"layer": layer, "cosine_sims": cosine_sims, "experiments": []}

    for scenario_key in SCENARIOS:
        prompt = prompts[scenario_key]
        print(f"\n--- {scenario_key} ---")
        experiment = {"scenario": scenario_key, "directions": []}

        # All directions to test: empathy + randoms
        all_dirs = [("empathy", None, empathy_dir_np)] + \
                   [("random", i, d) for i, d in enumerate(random_dirs)]

        for dir_type, dir_idx, dir_np in all_dirs:
            label = dir_type if dir_idx is None else f"{dir_type}_{dir_idx}"
            conditions = []
            for alpha in ALPHAS:
                print(f"  [{label}] alpha={alpha:+.1f} ", end="", flush=True)
                samples = []
                for s in range(NUM_SAMPLES):
                    c = generate_steered(model, tokenizer, prompt, dir_np, layer, alpha)
                    samples.append(c)
                    print(".", end="", flush=True)
                analysis = analyze_samples(samples)
                conditions.append({"alpha": alpha, "samples": samples, **analysis})
                fm = analysis["failure_mode_counts"]
                print(f"  coh={analysis['coherent_rate']:.2f}  len={analysis['avg_length']:.0f}  modes={fm}")

            experiment["directions"].append({
                "direction_type": dir_type, "direction_index": dir_idx,
                "conditions": conditions,
            })
        layer_result["experiments"].append(experiment)
    results["layer_results"].append(layer_result)

# Save
out_path = RESULTS_DIR / "dolphin_random_direction_control.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {out_path}")

In [ ]:
# ── Summary comparison table ───────────────────────────────────────────────
print(f"\n{'='*100}")
print("AGGREGATE COMPARISON: empathy direction vs mean-of-random-directions")
print(f"{'='*100}")
print(f"{'Scenario':<20} {'Alpha':>8} {'Emp.Coh':>9} {'Rnd.Coh(μ)':>11} "
      f"{'Emp.Len':>9} {'Rnd.Len(μ)':>11} {'Verdict'}")
print(f"{'─'*90}")

for layer_result in results["layer_results"]:
    for experiment in layer_result["experiments"]:
        scenario = experiment["scenario"]
        dirs = experiment["directions"]
        emp = next(d for d in dirs if d["direction_type"] == "empathy")
        rnds = [d for d in dirs if d["direction_type"] == "random"]

        for ci, ec in enumerate(emp["conditions"]):
            alpha = ec["alpha"]
            emp_coh = ec["coherent_rate"]
            emp_len = ec["avg_length"]

            rnd_cohs = [rd["conditions"][ci]["coherent_rate"] for rd in rnds]
            rnd_lens = [rd["conditions"][ci]["avg_length"] for rd in rnds]
            rnd_coh_mu = np.mean(rnd_cohs)
            rnd_len_mu = np.mean(rnd_lens)

            if abs(emp_coh - rnd_coh_mu) < 0.15:
                verdict = "SIMILAR → H2"
            elif emp_coh < rnd_coh_mu - 0.15:
                verdict = "EMPATHY WORSE → H3/H4"
            else:
                verdict = "RANDOM WORSE → unexpected"

            print(f"{scenario:<20} {alpha:>+8.1f} {emp_coh:>9.2f} {rnd_coh_mu:>11.2f} "
                  f"{emp_len:>9.1f} {rnd_len_mu:>11.1f} {verdict}")

print(f"\n{'='*100}")
print("SIMILAR → H2  : off-manifold instability (not empathy-specific)")
print("EMPATHY WORSE → H3/H4 : something specific about the empathy direction")
print(f"{'='*100}")

In [ ]:
# ── Inspect individual samples ─────────────────────────────────────────────
# Pick a specific alpha to compare raw outputs side by side
INSPECT_ALPHA = -5.0

for lr in results["layer_results"]:
    for exp in lr["experiments"]:
        print(f"\n{'='*80}")
        print(f"Layer {lr['layer']} / {exp['scenario']} / alpha={INSPECT_ALPHA}")
        print(f"{'='*80}")
        for d in exp["directions"]:
            label = d["direction_type"] if d["direction_index"] is None else f"{d['direction_type']}_{d['direction_index']}"
            cond = next(c for c in d["conditions"] if c["alpha"] == INSPECT_ALPHA)
            print(f"\n[{label}]  coherent={cond['coherent_rate']:.2f}  modes={cond['failure_mode_counts']}")
            for i, s in enumerate(cond["samples"]):
                preview = s[:150].replace('\n', ' ') if s else '<EMPTY>'
                print(f"  sample {i}: {preview}")

In [ ]:
# ── (Optional) Run full experiment: all 3 scenarios, 5 random dirs ─────────
# Uncomment and re-run the experiment cell above after changing these:
#
# SCENARIOS = ["food_delivery", "the_listener", "the_protector"]
# NUM_RANDOM_DIRS = 5
# NUM_SAMPLES = 3
# LAYERS = [8, 12, 16]  # all three layers

In [ ]:
# ── Download results ───────────────────────────────────────────────────────
from google.colab import files
files.download(str(out_path))